In [1]:
# ══════════════════════════════════════════════════════════
# CELDA 1 — Cargar IPD para exploración
# ══════════════════════════════════════════════════════════
# Este notebook documenta el proceso de selección de variables
# y dimensiones del IPD. No modifica ningún resultado —
# es únicamente análisis exploratorio y justificación metodológica.

from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

BASE    = '/content/drive/MyDrive/TFM_Seguridad_Vial'
outputs = f'{BASE}/outputs'

# Cargamos el IPD ya calculado y la tabla anual
ipd = pd.read_csv(f'{outputs}/IPD_completo.csv')
df  = pd.read_csv(f'{outputs}/tabla_anual_final.csv')

# Recalculamos variables derivadas necesarias para el análisis
df['acc_pond_km2']        = (df['acc_ponderado'] / df['area_km2']).round(4)
df['tasa_mortalidad']     = (df['acc_vru_mortales'] / df['poblacion'] * 100_000).round(4)
df['km_carril_km2']       = (df['km_carril_bici'] / df['area_km2']).round(4)
df['densidad_peatonal']   = (df['poblacion'] / df['area_km2']).round(2)
df['densidad_vulnerable'] = (
    (df['poblacion'] * (df['pct_mayores_65'] + df['pct_menores_15']) / 100)
    / df['area_km2']
).round(2)

# Tendencia interanual — necesaria para la matriz de correlación (celda 2)
def calcular_tendencia(df_input):
    resultados = []
    for distrito in df_input['distrito'].unique():
        d = df_input[df_input['distrito'] == distrito].sort_values('año')
        for año in [2021, 2022, 2023, 2024]:
            ventana = d[d['año'].between(año-2, año)]['acc_vru_total'].values
            if len(ventana) == 3:
                pendiente = np.polyfit([0,1,2], ventana, 1)[0]
            else:
                pendiente = np.nan
            resultados.append({
                'distrito': distrito,
                'año': año,
                'tendencia': round(float(pendiente), 4)
            })
    return pd.DataFrame(resultados)

tendencias = calcular_tendencia(df)
df = df.merge(tendencias, on=['distrito', 'año'], how='left')

# Filtramos 2021-2024 y añadimos dimensiones e IPD exportados
ipd_exp = df[df['año'].between(2021, 2024)].copy().reset_index(drop=True)
dims = ['dim_seguridad', 'dim_vulnerabilidad']

ipd_exp = ipd_exp.merge(
    ipd[['distrito', 'año', 'IPD', 'dim_seguridad', 'dim_vulnerabilidad']],
    on=['distrito', 'año'], how='left'
)

print(f'Datos cargados: {len(ipd_exp)} observaciones')
print(f'Años: {sorted(ipd_exp["año"].unique())}')
print(f'Nulos en tendencia: {ipd_exp["tendencia"].isnull().sum()}')
print(f'Columnas disponibles: {len(ipd_exp.columns)}')

Mounted at /content/drive
Datos cargados: 84 observaciones
Años: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Nulos en tendencia: 0
Columnas disponibles: 35


In [2]:
# ══════════════════════════════════════════════════════════
# CELDA 2 — Análisis de correlación entre variables brutas
# ══════════════════════════════════════════════════════════
# OBJETIVO: identificar variables redundantes (r > 0.90)
# y variables con causalidad inversa respecto a la accidentalidad.
# RESULTADO: base para las decisiones de exclusión de variables.

vars_check = [
    'acc_pond_km2', 'tasa_mortalidad', 'densidad_peatonal',
    'pct_mayores_65', 'densidad_vulnerable', 'pct_menores_15',
    'vel_media_vial', 'n_estaciones_bicimad', 'semaforos_km2',
    'pct_red_30kmh', 'km_carril_km2'
]

print('=== MATRIZ DE CORRELACIÓN — VARIABLES BRUTAS (2021-2024) ===')
print(ipd_exp[vars_check].corr().round(3).to_string())

print('\n=== HALLAZGOS CLAVE ===')
print('densidad_peatonal vs densidad_vulnerable : ',
      round(ipd_exp['densidad_peatonal'].corr(ipd_exp['densidad_vulnerable']),3),
      '→ double counting (>0.90) → eliminar densidad_peatonal')
print('pct_menores_15 vs acc_pond_km2           : ',
      round(ipd_exp['pct_menores_15'].corr(ipd_exp['acc_pond_km2']),3),
      '→ causalidad inversa → eliminar pct_menores_15')
print('vel_media_vial vs densidad_peatonal      : ',
      round(ipd_exp['vel_media_vial'].corr(ipd_exp['densidad_peatonal']),3),
      '→ mide suburbanidad, no riesgo → eliminar vel_media_vial')
print('n_estaciones_bicimad vs acc_pond_km2     : ',
      round(ipd_exp['n_estaciones_bicimad'].corr(ipd_exp['acc_pond_km2']),3),
      '→ mediada por densidad urbana → eliminar')

=== MATRIZ DE CORRELACIÓN — VARIABLES BRUTAS (2021-2024) ===
                      acc_pond_km2  tasa_mortalidad  densidad_peatonal  pct_mayores_65  densidad_vulnerable  pct_menores_15  vel_media_vial  n_estaciones_bicimad  semaforos_km2  pct_red_30kmh  km_carril_km2
acc_pond_km2                 1.000            0.197              0.871           0.175                0.784          -0.816          -0.674                 0.534          0.750          0.099          0.789
tasa_mortalidad              0.197            1.000              0.037           0.081               -0.005          -0.191          -0.194                 0.178         -0.044          0.165          0.058
densidad_peatonal            0.871            0.037              1.000           0.328                0.973          -0.813          -0.690                 0.297          0.931          0.024          0.813
pct_mayores_65               0.175            0.081              0.328           1.000                0.481    

In [3]:
# ══════════════════════════════════════════════════════════
# CELDA 3 — "Causalidad" inversa en variables de infraestructura
# ══════════════════════════════════════════════════════════
# OBJETIVO: verificar si semáforos y zonas 30 presentan
# causalidad inversa respecto a la accidentalidad.
# RESULTADO: justificación para eliminar la dimensión infraestructura.

df_2024 = ipd_exp[ipd_exp['año'] == 2024].copy()

print('=== pct_red_30kmh vs IPD 2024 ===')
print(df_2024[['distrito','pct_red_30kmh','IPD']]
      .sort_values('pct_red_30kmh', ascending=False).round(2).to_string(index=False))
print(f'\nCorrelación pct_red_30kmh vs acc_pond_km2: '
      f'{df_2024["pct_red_30kmh"].corr(df_2024["acc_pond_km2"]):.4f}')

print('\n=== semaforos_km2 vs IPD 2024 ===')
print(df_2024[['distrito','semaforos_km2','IPD']]
      .sort_values('semaforos_km2', ascending=False).round(2).to_string(index=False))
print(f'\nCorrelación semaforos_km2 vs acc_pond_km2: '
      f'{df_2024["semaforos_km2"].corr(df_2024["acc_pond_km2"]):.4f}')

print('\n=== CONCLUSIÓN ===')
print('Los distritos con más zonas 30 y más semáforos son los más céntricos')
print('y densos, que también tienen más accidentes absolutamente.')
print('Ambas variables están mediadas por densidad urbana → causalidad inversa.')
print('Decisión: eliminar dimensión Infraestructura del IPD.')
print('Los datos de infraestructura se mantienen en el dataset para Power BI.')

=== pct_red_30kmh vs IPD 2024 ===
           distrito  pct_red_30kmh   IPD
             CENTRO          73.36 58.54
              USERA          71.99 36.38
        CARABANCHEL          71.63 32.44
 PUENTE DE VALLECAS          70.89 45.16
             TETUÁN          69.27 53.11
         VILLAVERDE          66.89 18.19
             LATINA          64.37 37.35
            BARAJAS          63.26 18.45
FUENCARRAL-EL PARDO          60.65 22.99
SAN BLAS-CANILLEJAS          59.99 27.55
    MONCLOA-ARAVACA          59.90 32.93
          MORATALAZ          58.18 54.98
  VILLA DE VALLECAS          57.42  7.53
          SALAMANCA          54.05 55.68
           CHAMBERÍ          53.02 66.58
          HORTALEZA          50.73 26.78
             RETIRO          49.39 72.54
          CHAMARTÍN          46.02 56.26
          VICÁLVARO          44.79  8.41
         ARGANZUELA          41.36 47.65
      CIUDAD LINEAL          39.76 49.75

Correlación pct_red_30kmh vs acc_pond_km2: 0.1022

=== semaforo

In [4]:
# ══════════════════════════════════════════════════════════
# CELDA 4 — Correlación entre dimensiones finales
# ══════════════════════════════════════════════════════════
# OBJETIVO: verificar que las dos dimensiones finales son
# independientes entre sí (condición de no-redundancia).
# RESULTADO: correlación baja confirma independencia.

print('=== CORRELACIÓN ENTRE DIMENSIONES FINALES (2021-2024) ===')
print(ipd_exp[dims + ['IPD']].corr().round(3).to_string())

r_dims = ipd_exp['dim_seguridad'].corr(ipd_exp['dim_vulnerabilidad'])
print(f'\nCorrelación Seguridad vs Vulnerabilidad: {r_dims:.4f}')
if abs(r_dims) < 0.30:
    print('✅ Dimensiones independientes — no hay redundancia entre ellas')
else:
    print('⚠️ Correlación moderada — revisar')

print('\n=== CORRELACIÓN DIMENSIONES CON IPD ===')
for dim in dims:
    r = ipd_exp['IPD'].corr(ipd_exp[dim])
    print(f'{dim:<25}: {r:.4f}')

=== CORRELACIÓN ENTRE DIMENSIONES FINALES (2021-2024) ===
                    dim_seguridad  dim_vulnerabilidad    IPD
dim_seguridad               1.000               0.330  0.746
dim_vulnerabilidad          0.330               1.000  0.875
IPD                         0.746               0.875  1.000

Correlación Seguridad vs Vulnerabilidad: 0.3302
⚠️ Correlación moderada — revisar

=== CORRELACIÓN DIMENSIONES CON IPD ===
dim_seguridad            : 0.7457
dim_vulnerabilidad       : 0.8751


In [5]:
# ══════════════════════════════════════════════════════════
# CELDA 5 — Descomposición IPD 2024 por dimensión
# ══════════════════════════════════════════════════════════
# OBJETIVO: verificar que el ranking tiene sentido geográfico
# y que cada dimensión contribuye de forma coherente.

df_2024 = ipd_exp[ipd_exp['año'] == 2024].copy()
df_2024['contribucion_seg'] = df_2024['dim_seguridad'] * 0.55
df_2024['contribucion_vul'] = df_2024['dim_vulnerabilidad'] * 0.45

print('=== DESCOMPOSICIÓN IPD 2024 ===')
cols = ['distrito','IPD','contribucion_seg','contribucion_vul',
        'acc_pond_km2','tasa_mortalidad','pct_mayores_65','densidad_vulnerable']
print(df_2024[cols].sort_values('IPD', ascending=False).round(2).to_string(index=False))

=== DESCOMPOSICIÓN IPD 2024 ===
           distrito   IPD  contribucion_seg  contribucion_vul  acc_pond_km2  tasa_mortalidad  pct_mayores_65  densidad_vulnerable
             RETIRO 72.54             31.14             41.40         25.78             1.66           26.81              8336.61
           CHAMBERÍ 66.58             27.10             39.48         41.24             0.70           23.97             10266.22
             CENTRO 58.54             42.22             16.31         56.79             1.37           15.68              6209.34
          CHAMARTÍN 56.26             25.11             31.14         19.83             1.34           23.75              5901.39
          SALAMANCA 55.68             18.01             37.67         35.81             0.00           23.80              9447.83
          MORATALAZ 54.98             19.56             35.42         10.66             1.04           26.19              5693.63
             TETUÁN 53.11             25.50             27

In [6]:
# ══════════════════════════════════════════════════════════
# CELDA 6 — Justificación de acc_pond_km2 vs tasa por habitante
# ══════════════════════════════════════════════════════════
# OBJETIVO: verificar si acc_pond_km2 y tasa_accidentalidad
# son redundantes (r > 0.90) o miden cosas distintas.
# RESULTADO: r = 0.78 → no redundantes → se mantiene acc_pond_km2
# por su justificación conceptual para intervenciones físicas.

df_2024 = ipd_exp[ipd_exp['año'] == 2024].copy()
df_2024['tasa_accidentalidad'] = (
    df_2024['acc_ponderado'] / df_2024['poblacion'] * 100_000
).round(3)

r1 = df_2024['acc_pond_km2'].corr(df_2024['tasa_accidentalidad'])
r2 = df_2024['tasa_accidentalidad'].corr(df_2024['tasa_mortalidad'])

print(f'Correlación acc_pond_km2 vs tasa_accidentalidad : {r1:.4f}')
print(f'Correlación tasa_accidentalidad vs tasa_mortalidad: {r2:.4f}')

print('\n=== CONCLUSIÓN ===')
print(f'r = {r1:.2f} < 0.90 → no redundantes (Nardo et al., 2008)')
print('acc_pond_km2 mide concentración espacial del riesgo')
print('tasa_mortalidad ya actúa como corrector de equidad poblacional')
print('Decisión: mantener acc_pond_km2 como métrica principal de accidentalidad')

# Comparación visual de rankings
mn1, mx1 = df_2024['acc_pond_km2'].min(), df_2024['acc_pond_km2'].max()
mn2, mx2 = df_2024['tasa_accidentalidad'].min(), df_2024['tasa_accidentalidad'].max()
df_2024['rank_km2']  = ((df_2024['acc_pond_km2'] - mn1)/(mx1-mn1)*100).round(1)
df_2024['rank_tasa'] = ((df_2024['tasa_accidentalidad'] - mn2)/(mx2-mn2)*100).round(1)
df_2024['diferencia'] = (df_2024['rank_km2'] - df_2024['rank_tasa']).round(1)

print('\n=== RANKING COMPARATIVO: densidad/km² vs tasa/habitante ===')
print(df_2024[['distrito','rank_km2','rank_tasa','diferencia']]
      .sort_values('rank_km2', ascending=False).to_string(index=False))

Correlación acc_pond_km2 vs tasa_accidentalidad : 0.7711
Correlación tasa_accidentalidad vs tasa_mortalidad: 0.4702

=== CONCLUSIÓN ===
r = 0.77 < 0.90 → no redundantes (Nardo et al., 2008)
acc_pond_km2 mide concentración espacial del riesgo
tasa_mortalidad ya actúa como corrector de equidad poblacional
Decisión: mantener acc_pond_km2 como métrica principal de accidentalidad

=== RANKING COMPARATIVO: densidad/km² vs tasa/habitante ===
           distrito  rank_km2  rank_tasa  diferencia
             CENTRO     100.0      100.0         0.0
           CHAMBERÍ      72.3       52.3        20.0
          SALAMANCA      62.6       48.0        14.6
             TETUÁN      56.9       30.8        26.1
         ARGANZUELA      46.0       33.9        12.1
             RETIRO      44.7       40.4         4.3
          CHAMARTÍN      34.1       43.5        -9.4
              USERA      33.4       27.2         6.2
 PUENTE DE VALLECAS      31.5       32.5        -1.0
      CIUDAD LINEAL      25.2  

In [7]:
# ══════════════════════════════════════════════════════════
# CELDA 7 — Validación empírica de las variables del IPD
# ══════════════════════════════════════════════════════════
# OBJETIVO: verificar que las variables seleccionadas tienen
# correlaciones coherentes con la accidentalidad real observada,
# y que no son redundantes entre sí.
# IMPORTANTE: estas correlaciones NO se usan para derivar pesos
# (el IPD es descriptivo y sus pesos son normativos, justificados
# por literatura). Se usan únicamente como validación de coherencia
# interna del diseño del índice.
# Referencia metodológica: Nardo et al. (2008), cap. 7.

df_val = ipd_exp[ipd_exp['año'] == 2024].copy()

# ── Correlación de cada variable con el outcome externo ───────────────────
outcome = 'acc_ponderado'
vars_ipd = ['acc_pond_km2', 'tasa_mortalidad', 'tendencia',
            'pct_mayores_65', 'densidad_vulnerable']

print(f'=== CORRELACIÓN DE VARIABLES IPD CON OUTCOME EXTERNO ({outcome}) ===')
print(f'(Valores más altos indican mayor relación con la accidentalidad real)\n')
corrs = {}
for v in vars_ipd:
    r = df_val[outcome].corr(df_val[v])
    corrs[v] = round(r, 4)
    print(f'{v:<30}: {r:+.4f}')

# ── Interpretación por dimensión ──────────────────────────────────────────
print('\n=== INTERPRETACIÓN ==='  )
print('\nDIMENSIÓN SEGURIDAD:')
print(f'  acc_pond_km2    ({corrs["acc_pond_km2"]:+.4f}): correlación alta y directa ✅')
print(f'  tasa_mortalidad ({corrs["tasa_mortalidad"]:+.4f}): correlación moderada y directa ✅')
print(f'  tendencia       ({corrs["tendencia"]:+.4f}): correlación moderada ✅')
print('  → Las tres variables de Seguridad correlacionan positivamente')
print('    con la accidentalidad real. No hay causalidad inversa.')

print('\nDIMENSIÓN VULNERABILIDAD:')
print(f'  pct_mayores_65      ({corrs["pct_mayores_65"]:+.4f}): correlación baja con outcome absoluto')
print(f'  densidad_vulnerable ({corrs["densidad_vulnerable"]:+.4f}): correlación moderada ✅')
print('  → La baja correlación de pct_mayores_65 con acc_ponderado')
print('    no invalida su inclusión: mide riesgo potencial de gravedad,')
print('    no frecuencia de accidentes. Respaldado por Gálvez-Pérez et al. (2022).')
print('  → La Vulnerabilidad corrige el ranking hacia distritos con mayor')
print('    probabilidad de consecuencias graves, función conceptualmente')
print('    distinta de la dimensión Seguridad.')

# ── Verificación de no-redundancia entre variables finales ────────────────
print('\n=== VERIFICACIÓN DE NO-REDUNDANCIA (r < 0.90) ===')
import itertools
for v1, v2 in itertools.combinations(vars_ipd, 2):
    r = abs(df_val[v1].corr(df_val[v2]))
    estado = '✅' if r < 0.90 else '⚠️ REDUNDANTE'
    print(f'{v1:<30} vs {v2:<30}: {r:.4f} {estado}')

# ── Conclusión ────────────────────────────────────────────────────────────
print('\n=== CONCLUSIÓN ==='  )
print('Las variables de Seguridad presentan correlaciones coherentes')
print('con la accidentalidad real (0.29–0.64), confirmando que miden')
print('aspectos distintos del mismo fenómeno sin redundancia.')
print('Las variables de Vulnerabilidad tienen correlaciones bajas con')
print('el outcome absoluto por diseño — miden riesgo estructural,')
print('no frecuencia histórica. Los pesos del IPD son normativos,')
print('justificados por literatura, y validados mediante análisis')
print('de sensibilidad (rho ≥ 0.96 en todos los escenarios razonables).')

=== CORRELACIÓN DE VARIABLES IPD CON OUTCOME EXTERNO (acc_ponderado) ===
(Valores más altos indican mayor relación con la accidentalidad real)

acc_pond_km2                  : +0.6685
tasa_mortalidad               : +0.4650
tendencia                     : +0.4611
pct_mayores_65                : +0.0113
densidad_vulnerable           : +0.4533

=== INTERPRETACIÓN ===

DIMENSIÓN SEGURIDAD:
  acc_pond_km2    (+0.6685): correlación alta y directa ✅
  tasa_mortalidad (+0.4650): correlación moderada y directa ✅
  tendencia       (+0.4611): correlación moderada ✅
  → Las tres variables de Seguridad correlacionan positivamente
    con la accidentalidad real. No hay causalidad inversa.

DIMENSIÓN VULNERABILIDAD:
  pct_mayores_65      (+0.0113): correlación baja con outcome absoluto
  densidad_vulnerable (+0.4533): correlación moderada ✅
  → La baja correlación de pct_mayores_65 con acc_ponderado
    no invalida su inclusión: mide riesgo potencial de gravedad,
    no frecuencia de accidentes. Res